In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import json
import os
import gc
# import mediapipe as mp
# mp_holistic = mp.solutions.holistic
from tensorflow.keras import layers, models, optimizers

In [ ]:
# Prevent OpenCV from competing with TensorFlow's multi-threading,
# which can cause graph execution errors in tf.data.Datasets
cv2.setNumThreads(0)

# A predefined subset of 35 critical facial landmarks (eyes, eyebrows, mouth)
# This reduces the face feature bloat from 1404 values down to 105 values.
SELECTED_FACE_INDICES = [
    # Lips
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95,
    # Left Eye & Eyebrow
    33, 133, 159, 145, 46, 52, 53,
    # Right Eye & Eyebrow
    362, 263, 386, 374, 276, 282, 283
]

def extract_keypoints(results):
    """Extracts, normalizes, and flattens landmarks from MediaPipe Holistic results."""

    # POINT B: Define an anchor point for spatial normalization.
    # We use the Pose Nose (landmark 0) if available.
    if results.pose_landmarks:
        anchor_x = results.pose_landmarks.landmark[0].x
        anchor_y = results.pose_landmarks.landmark[0].y
        anchor_z = results.pose_landmarks.landmark[0].z
    else:
        # Fallback if no pose is detected at all
        anchor_x, anchor_y, anchor_z = 0.0, 0.0, 0.0

    # Pose: 33 landmarks. Normalize x, y, z, but keep visibility untouched.
    if results.pose_landmarks:
        pose = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z, res.visibility]
                         for res in results.pose_landmarks.landmark]).flatten()
    else:
        pose = np.zeros(33 * 4)

    # Face (POINT A): Only extract the 35 indices defined above and normalize them.
    if results.face_landmarks:
        face = np.array([[results.face_landmarks.landmark[i].x - anchor_x,
                          results.face_landmarks.landmark[i].y - anchor_y,
                          results.face_landmarks.landmark[i].z - anchor_z]
                         for i in SELECTED_FACE_INDICES]).flatten()
    else:
        face = np.zeros(len(SELECTED_FACE_INDICES) * 3)

    # Left Hand: 21 landmarks. Normalize x, y, z.
    if results.left_hand_landmarks:
        lh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # Right Hand: 21 landmarks. Normalize x, y, z.
    if results.right_hand_landmarks:
        rh = np.array([[res.x - anchor_x, res.y - anchor_y, res.z - anchor_z]
                       for res in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])